# Evaluasi split test: produksi vs MuQ saja

Buka dari root **SongMap**; memakai Python `.venv-songformer311` di sibling Section-Annotations.

Pipeline `muq` + konteks lolos di validation (0,8027 / 0,7674 vs produksi 0,7938 / 0,7755). Validation sudah dipakai untuk memilih, jadi keputusan akhir diambil **sekali** di split test (34 lagu). Tiga pipeline dinilai pada section test yang identik:

| Pipeline | Head | Konteks |
|---|---|---|
| `production` | `worship-v3` | konteks lama (epoch 5) |
| `all4` | MusicFM + MuQ, window 120 | dilatih ulang |
| `muq` | MuQ saja, window 120 | dilatih ulang |

Sebelum inferensi test, setiap pipeline harus mereproduksi angka validation-nya, lalu model, aturan dan inventaris test dikunci di `selection.json`. Aturan (tetap): `muq` lolos bila akurasi dan macro F1 test masing-masing turun maksimal 0,01 dibanding `production`. Tidak ada fitting atau pemilihan pada test. Jangan melatih ulang berdasarkan hasil ini.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, json
assert Path('scripts/evaluate_variant_test.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=Path('../Section-Annotations').resolve()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
PRODUCTION_HEAD=SONGSTRUCT_ROOT/'outputs.local/finetuning/runs/worship-v3/best.pt'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
assert PRODUCTION_HEAD.is_file(), f'worship-v3 tidak ditemukan: {PRODUCTION_HEAD}'
# Konteks produksi: run validation lengkap yang memilih epoch 5 (bukan dipilih dari skor test).
runs=[p.parent.resolve() for p in Path('outputs.local').glob('section-context-*/comparison.json')
      if (lambda r: r.get('full_validation') and r.get('split')=='val' and r.get('best_epoch')==5)(json.loads(p.read_text()))]
assert len(runs)==1, f'Harus tepat satu run konteks produksi epoch 5, ditemukan: {runs}'
PRODUCTION_CONTEXT_RUN=runs[0]
VARIANT_CONTEXT=sorted(Path('outputs.local').glob('stream-ablation-w120-*'))[-1].resolve()/'context'
assert json.loads((VARIANT_CONTEXT/'muq'/'comparison.json').read_text())['comparison']['verdict']=='candidate'
OUTPUT=Path('outputs.local')/('variant-pipeline-test-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
CACHE=Path('outputs.local')/'variant-context-cache'
print('Konteks produksi:',PRODUCTION_CONTEXT_RUN)
print('Konteks varian:',VARIANT_CONTEXT)
print('Output:',OUTPUT.resolve())

In [ ]:
OUTPUT.mkdir(parents=True,exist_ok=True)
log=OUTPUT/'test.log'
command=[str(RUN_PYTHON),'-u','scripts/evaluate_variant_test.py','--songstruct-root',str(SONGSTRUCT_ROOT),
    '--production-head',str(PRODUCTION_HEAD),'--production-context-run',str(PRODUCTION_CONTEXT_RUN),
    '--variant-context-dir',str(VARIANT_CONTEXT),'--cache-dir',str(CACHE),'--output-dir',str(OUTPUT),'--device','mps']
with log.open('w',encoding='utf-8') as handle:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='');handle.write(line);handle.flush()
        status=process.wait()
    except KeyboardInterrupt:
        process.terminate();process.wait();raise
if status:
    raise RuntimeError(f'Gagal ({status}); lihat {log.resolve()}')
print('Bagikan:',(OUTPUT/'test-comparison.json').resolve())

Bila `muq` lolos di test, langkah berikutnya adalah ekspor Core ML (head input 2048 palette6, konteks FP32) dan probe iPhone **tanpa MusicFM**, untuk mengukur ukuran dan waktu sebenarnya. Bila ditolak, kedua encoder tetap dipakai.